In [ ]:
def create_ford_database(db_name="ford_inventory.db"):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    # Create the vehicles table
    create_table_query = """
    CREATE TABLE IF NOT EXISTS vehicles (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        model TEXT NOT NULL,
        price REAL NOT NULL,
        year INTEGER NOT NULL,
        edition TEXT DEFAULT 'Normal',
        color TEXT NOT NULL,
        country TEXT NOT NULL,
        available_city TEXT NOT NULL,
        quantity_available INTEGER NOT NULL
    )
    """
    
    cursor.execute(create_table_query)
    conn.commit()
    
    print(f"Database '{db_name}' created successfully with 'vehicles' table")
    return conn

def populate_ford_database(conn, inventory_data):
    cursor = conn.cursor()
    
    # Clear existing data
    cursor.execute("DELETE FROM vehicles")
    
    # Insert inventory data
    insert_query = """
    INSERT INTO vehicles (id, model, price, year, edition, color, country, available_city, quantity_available)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    
    for vehicle_id, vehicle_data in inventory_data.items():
        cursor.execute(insert_query, (
            vehicle_id,
            vehicle_data['model'],
            vehicle_data['price'],
            vehicle_data['year'],
            vehicle_data['edition'],
            vehicle_data['color'],
            vehicle_data['country'],
            vehicle_data['available_city'],
            vehicle_data['quantity_available']
        ))
    
    conn.commit()
    print(f"Inserted {len(inventory_data)} vehicles into the database")


In [ ]:
def filter_cars_from_db(conn, model=None, max_price=None, city=None, country=None, 
                         color=None, year=None, edition=None):
    cursor = conn.cursor()
    # Build the query dynamically
    query = "SELECT * FROM vehicles WHERE 1=1"
    params = []
    
    if model:
        query += " AND model = ?"
        params.append(model)
    if max_price:
        query += " AND price <= ?"
        params.append(max_price)
    if city:
        query += " AND available_city = ?"
        params.append(city)
    if country:
        query += " AND country = ?"
        params.append(country)
    if color:
        query += " AND color = ?"
        params.append(color)
    if year:
        query += " AND year = ?"
        params.append(year)
    if edition:
        query += " AND edition = ?"
        params.append(edition)
    
    cursor.execute(query, params)
    rows = cursor.fetchall()
    
    # Convert rows to dictionaries
    columns = [description[0] for description in cursor.description]
    results = []
    for row in rows:
        vehicle_dict = dict(zip(columns, row))
        results.append(vehicle_dict)
    
    return results


In [ ]:
ford_db = create_ford_database()
cursor = ford_db.cursor()
cursor.execute("SELECT COUNT(*) FROM vehicles")
row_count = cursor.fetchone()[0]
if row_count == 0: populate_ford_database(ford_db, ford_inventory)
 
# Get row count

print(f"Total vehicles in database: {row_count}")

# Test the database query
test_results = filter_cars_from_db(ford_db, model="Mustang")
print(f"Found {len(test_results)} Mustangs:")
for vehicle in test_results:
    print(f"  - {vehicle['model']} {vehicle['edition']} ({vehicle['color']}) - ${vehicle['price']}")

In [ ]:
def add_vehicle_to_database(conn, vehicle_id, model, price, year, edition, color, 
                           country, available_city, quantity_available):
    """
    Add a single vehicle to the Ford database.
    
    Args:
        conn: SQLite database connection
        vehicle_id (str): Unique vehicle identifier
        model (str): Vehicle model name
        price (float): Vehicle price
        year (int): Vehicle year
        edition (str): Vehicle edition
        color (str): Vehicle color
        country (str): Country where available
        available_city (str): City where available
        quantity_available (int): Available quantity
    """
    cursor = conn.cursor()
    
    insert_query = """
    INSERT OR REPLACE INTO vehicles (id, model, price, year, edition, color, country, available_city, quantity_available)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    
    cursor.execute(insert_query, (
        vehicle_id,
        model,
        price,
        year,
        edition,
        color,
        country,
        available_city,
        quantity_available
    ))
    
    conn.commit()
    print(f"Added/Updated vehicle {vehicle_id}: {model} {edition} ({color}) - ${price}")

    

In [ ]:
insert_function = {
    "name": "add_vehicle_to_database",
    "description": (
        "Add a new vehicle to the Ford inventory database. "
        "Only the model name is required; other fields are optional."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "model": {
                "type": "string",
                "description": "Vehicle model name (required), e.g. Mustang, F-150, Explorer"
            },
            "price": {
                "type": "number",
                "description": "Vehicle price (optional)"
            },
            "year": {
                "type": "integer",
                "description": "Vehicle year (optional)"
            },
            "edition": {
                "type": "string",
                "description": "Vehicle edition (optional), e.g. Normal, GT, Raptor"
            },
            "color": {
                "type": "string",
                "description": "Vehicle color (optional)"
            },
            "country": {
                "type": "string",
                "description": "Country where vehicle is available (optional)"
            },
            "available_city": {
                "type": "string",
                "description": "City where vehicle is available (optional)"
            },
            "quantity_available": {
                "type": "integer",
                "description": "Available quantity (optional)"
            }
        },
        "required": ["model"],
        "additionalProperties": False
    },
    "strict": False
}